In [1]:
!pip install earthaccess asf_search nisar-pytools pystac-client planetary-computer rasterio geopandas xarray dask requests
!pip install rioxarray rasterio pandas numpy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s  0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37/37 [planetary-computer]


In [2]:
!rm -rf /kaggle/working/*
print("Working directory cleared!")

Working directory cleared!


In [3]:
import os
import requests
import json
import numpy as np
import h5py
import dask.array as da
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import earthaccess
from sklearn.cluster import KMeans
from kaggle_secrets import UserSecretsClient

# 1. AUTHENTICATION
user_secrets = UserSecretsClient()
os.environ["EARTHDATA_USERNAME"] = user_secrets.get_secret("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = user_secrets.get_secret("EARTHDATA_PASSWORD")
BHOONIDHI_USER = user_secrets.get_secret("BHOONIDHI_USER")
BHOONIDHI_PASS = user_secrets.get_secret("BHOONIDHI_PASS")

earthaccess.login(strategy="environment")

bhoonidhi_auth_url = "https://bhoonidhi-api.nrsc.gov.in/auth/token"
auth_payload = {"userId": BHOONIDHI_USER, "password": BHOONIDHI_PASS, "grant_type": "password"}
headers = {"Content-Type": "application/json"}
response = requests.post(bhoonidhi_auth_url, json=auth_payload, headers=headers)
if response.status_code == 200:
    headers["Authorization"] = f"Bearer {response.json().get('access_token')}"
    print("Bhoonidhi Auth Successful!")

# 2. TARGET DATABASE (Added 'patch_deg' to define the dense center crop size)
REGIONS = {
    "assam": {"bbox": (91.50, 26.00, 92.50, 26.50), "platform": "ISRO_S_Band", "api": "bhoonidhi", "epsg": 32646, "patch_deg": 0.30},
    "amazon": {"bbox": (-62.50, -3.80, -61.00, -3.00), "platform": "NASA_L_Band", "api": "earthaccess", "epsg": 32720, "patch_deg": 0.12},
    "california": {"bbox": (-120.5, 36.2, -119.0, 37.0), "platform": "NASA_L_Band", "api": "earthaccess", "epsg": 32611, "patch_deg": 0.12},
    "sundarbans": {"bbox": (89.0, 21.5, 90.0, 22.0), "platform": "NASA_L_Band", "api": "earthaccess", "epsg": 32645, "patch_deg": 0.12}
}

start_date = "2026-07-01T00:00:00Z"
end_date = "2026-08-30T00:00:00Z"
export_dir = "/kaggle/working/frontend_assets"
os.makedirs(export_dir, exist_ok=True)
json_path = os.path.join(export_dir, "scientist_timeseries.json")

Bhoonidhi Auth Successful!


In [4]:
def process_and_export(ds, region_id, platform_name, patch_deg):
    print(f"Reprojecting {region_id.upper()} ({platform_name})...")
    x_name = 'x' if 'x' in ds.coords else 'xCoordinates'
    y_name = 'y' if 'y' in ds.coords else 'yCoordinates'
    ds = ds.rename({x_name: 'x', y_name: 'y'})
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")
    
    native_epsg = REGIONS[region_id]["epsg"]
    ds = ds.rio.write_crs(f"EPSG:{native_epsg}")
    
    # Downsample slightly to save RAM
    ds_small = ds.isel(x=slice(None, None, 4), y=slice(None, None, 4))
    ds_latlon = ds_small.rio.reproject("EPSG:4326")
    
    # ---------------------------------------------------------
    # FIX: DENSE CENTER PATCH EXTRACTION
    # ---------------------------------------------------------
    # Instead of slicing via a massive box that captures 80% white space, 
    # we locate the dead center of the flight swath and extract a perfect, solid square.
    
    valid_x = ds_latlon.x.values
    valid_y = ds_latlon.y.values
    center_lon = valid_x[len(valid_x) // 2]
    center_lat = valid_y[len(valid_y) // 2]
    
    min_lon = center_lon - (patch_deg / 2)
    max_lon = center_lon + (patch_deg / 2)
    
    # Latitude arrays in GeoTIFFs usually go from high to low (descending)
    if valid_y[0] > valid_y[-1]:
        max_lat_bound = center_lat + (patch_deg / 2)
        min_lat_bound = center_lat - (patch_deg / 2)
    else:
        max_lat_bound = center_lat - (patch_deg / 2)
        min_lat_bound = center_lat + (patch_deg / 2)

    print(f"[{region_id}] Extracting solid dense patch ({patch_deg}° x {patch_deg}°) at {center_lon:.2f}, {center_lat:.2f}")
    
    cropped_ds = ds_latlon.sel(x=slice(min_lon, max_lon), y=slice(max_lat_bound, min_lat_bound)).compute()
    
    if cropped_ds.x.size == 0 or cropped_ds.y.size == 0:
        print(f"[{region_id}] Center crop empty, reverting to full scene.")
        cropped_ds = ds_latlon.compute()
        
    co_pol = next((b for b in ['HHHH', 'VVVV', 'HH', 'VV'] if b in cropped_ds.data_vars), None)
    cross_pol = next((b for b in ['HVHV', 'VHVH', 'HV', 'VH'] if b in cropped_ds.data_vars), None)
        
    if co_pol and cross_pol:
        hh_real = np.abs(cropped_ds[co_pol])
        hv_real = np.abs(cropped_ds[cross_pol])
        
        cropped_ds['HHHH_dB'] = 10 * np.log10(hh_real.where(hh_real > 0) + 1e-10)
        cropped_ds['RVI'] = 4 * hv_real / (hh_real + hv_real + 1e-10)
        
        print(f"Running ML Clustering for {region_id}...")
        hh_flat = cropped_ds['HHHH_dB'].values.flatten()
        rvi_flat = cropped_ds['RVI'].values.flatten()
        
        valid_mask = ~np.isnan(hh_flat) & ~np.isnan(rvi_flat) & np.isfinite(hh_flat) & np.isfinite(rvi_flat)
        features = np.column_stack((hh_flat[valid_mask], rvi_flat[valid_mask]))
        
        if len(features) > 0:
            kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
            predictions = kmeans.fit_predict(features)
            ml_map = np.full(hh_flat.shape, np.nan)
            ml_map[valid_mask] = predictions
            cropped_ds['ML_Class'] = (("y", "x"), ml_map.reshape(cropped_ds['HHHH_dB'].shape))
            
            # Matplotlib Vis
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            cropped_ds['RVI'].plot(cmap='viridis', vmin=0, vmax=1.2)
            plt.title(f'{region_id.upper()} {platform_name} RVI\n(Dense Center Patch)')
            plt.axis('off')
            plt.subplot(1, 2, 2)
            cropped_ds['ML_Class'].plot(cmap='Set1')
            plt.title('ML Surface Segmentation')
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(export_dir, f"Vis_{region_id}.png"), dpi=150, bbox_inches='tight')
            plt.close()
            
            # Exact TIF Exports for Frontend Slider
            cropped_ds = cropped_ds.rio.write_crs("EPSG:4326")
            cropped_ds['HHHH_dB'].rio.to_raster(os.path.join(export_dir, f"{region_id}_raw_sar.tif"), driver="COG", compress="DEFLATE")
            cropped_ds['ML_Class'].rio.to_raster(os.path.join(export_dir, f"{region_id}_processed_kmeans.tif"), driver="COG", compress="DEFLATE")
            
            # Telemetry JSON
            mean_rvi = float(cropped_ds['RVI'].mean().values)
            mean_hh = float(cropped_ds['HHHH_dB'].mean().values)
            dashboard_data = {
                "region": region_id,
                "platform": platform_name,
                "mean_rvi": round(mean_rvi, 3),
                "mean_roughness_db": round(mean_hh, 2)
            }
            
            timeseries = []
            if os.path.exists(json_path):
                with open(json_path, 'r') as f: timeseries = json.load(f)
            timeseries.append(dashboard_data)
            with open(json_path, 'w') as f: json.dump(timeseries, f, indent=4)
            print(f"Exported {region_id} successfully.")
    else:
        print(f"[{region_id}] Missing required polarimetry bands. Found: {list(cropped_ds.data_vars.keys())}")

In [5]:
# 3. EXECUTION LOOP
for region_id, info in REGIONS.items():
    print(f"\n--- Fetching Data for {region_id.upper()} ---")
    if info["api"] == "earthaccess":
        results = earthaccess.search_data(
            short_name="NISAR_L2_GCOV_PROVISIONAL_V1",
            bounding_box=info["bbox"],
            temporal=(start_date, end_date),
            count=1
        )
        if results:
            print(f"[{region_id}] Downloading file locally to bypass CloudFront 403 restrictions...")
            # FIX: Download the file locally instead of HTTP streaming
            downloaded_files = earthaccess.download(results, local_path="/kaggle/working/")
            
            if downloaded_files:
                file_path = downloaded_files[0]
                try:
                    ds = xr.open_dataset(file_path, engine='h5netcdf', group='/science/LSAR/GCOV/grids/frequencyA', backend_kwargs={'phony_dims': 'sort'})
                    process_and_export(ds, region_id, info["platform"], info["patch_deg"])
                    ds.close()
                except Exception as e:
                    print(f"Error processing {region_id}: {e}")
                
                # Cleanup to save Kaggle disk space
                if os.path.exists(file_path): 
                    os.remove(file_path)
                
    elif info["api"] == "bhoonidhi":
        search_payload = {"collections": ["NISAR_SSAR_GCOV", "NISAR_SSAR-Beta_GCOV"], "bbox": info["bbox"], "datetime": f"{start_date}/{end_date}", "limit": 1}
        response = requests.post("https://bhoonidhi-api.nrsc.gov.in/data/search", headers=headers, json=search_payload)
        features = response.json().get("features", [])
        
        if not features:
            print(f"[{region_id}] Initial search empty. Expanding parameters...")
            search_payload.pop("bbox", None)
            search_payload.pop("datetime", None)
            response = requests.post("https://bhoonidhi-api.nrsc.gov.in/data/search", headers=headers, json=search_payload)
            features = response.json().get("features", [])[:1]
            
        if features:
            feature_id = features[0].get("id")
            collection_id = features[0].get("collection")
            file_path = f"/kaggle/working/temp_{region_id}.h5"
            download_url = f"https://bhoonidhi-api.nrsc.gov.in/download?id={feature_id}&collection={collection_id}"
            
            with requests.get(download_url, headers=headers, stream=True) as r:
                if r.status_code == 200:
                    with open(file_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
            try:
                with h5py.File(file_path, 'r') as f:
                    freq_a = f['/science/SSAR/GCOV/grids/frequencyA']
                    y_coords, x_coords = freq_a['yCoordinates'][:], freq_a['xCoordinates'][:]
                    
                    available_bands = [b for b in freq_a.keys() if 'HH' in b or 'VV' in b or 'HV' in b or 'VH' in b]
                    if not available_bands:
                         available_bands = [list(freq_a.keys())[0]]
                    
                    data_vars = {b: (["y", "x"], da.from_array(freq_a[b], chunks=(2000, 2000))) for b in available_bands}
                    ds = xr.Dataset(data_vars, coords={"y": y_coords, "x": x_coords})
                    process_and_export(ds, region_id, info["platform"], info["patch_deg"])
                    ds.close()
            except Exception as e:
                print(f"Error processing {region_id}: {e}")
            if os.path.exists(file_path): os.remove(file_path)

print("\nAll regions processed. Files ready in /kaggle/working/frontend_assets/")


--- Fetching Data for ASSAM ---
Reprojecting ASSAM (ISRO_S_Band)...
[assam] Extracting solid dense patch (0.3° x 0.3°) at 91.80, 26.73
Running ML Clustering for assam...
Exported assam successfully.

--- Fetching Data for AMAZON ---
[amazon] Downloading file locally to bypass CloudFront 403 restrictions...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Reprojecting AMAZON (NASA_L_Band)...
[amazon] Extracting solid dense patch (0.12° x 0.12°) at -63.26, -3.08
Running ML Clustering for amazon...
Exported amazon successfully.

--- Fetching Data for CALIFORNIA ---
[california] Downloading file locally to bypass CloudFront 403 restrictions...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Reprojecting SUNDARBANS (NASA_L_Band)...
[sundarbans] Extracting solid dense patch (0.12° x 0.12°) at 84.72, 22.77
Running ML Clustering for sundarbans...
Exported sundarbans successfully.

All regions processed. Files ready in /kaggle/working/frontend_assets/


In [6]:
import shutil
import requests
import os
from IPython.display import FileLink, display

# 1. Name of the zip file (without the .zip extension here)
zip_name = 'nisar_frontend_assets'
zip_filename = f"{zip_name}.zip"

# 2. The folder you want to zip
folder_to_zip = '/kaggle/working/frontend_assets' 

# 3. Create the zip file
shutil.make_archive(zip_name, 'zip', folder_to_zip)
print(f"Folder successfully zipped into {zip_filename}!")

# 4. Upload to robust cloud storage for background execution retrieval
print("Uploading to cloud storage (transfer.sh)...")
try:
    with open(zip_filename, 'rb') as f:
        # transfer.sh takes a PUT request and returns the URL as plain text
        response = requests.put(f'https://transfer.sh/{zip_filename}', data=f)
        
    if response.status_code == 200:
        download_url = response.text.strip()
        print(f"✅ Success! Save this link to download your file tomorrow: {download_url}")
    else:
        print(f"❌ Upload failed. Status code: {response.status_code}")
        print(f"Server response: {response.text}")
except Exception as e:
    print(f"❌ Upload error: {e}")

# 5. Generate a clickable download link (useful only if you run it interactively)
display(FileLink(zip_filename))

Folder successfully zipped into nisar_frontend_assets.zip!
Uploading to cloud storage (transfer.sh)...
❌ Upload error: HTTPSConnectionPool(host='transfer.sh', port=443): Max retries exceeded with url: /nisar_frontend_assets.zip (Caused by NewConnectionError("HTTPSConnection(host='transfer.sh', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))


/kaggle/working/nisar_frontend_assets.zip